# 17 — MIDI 2026: Self-verification-only Calibration Baseline (v1)

**Hakem 2 talebi:** `raw_verification` değerini yalnızca kendisi üzerinden kalibre etmek, ardından 8 özellikli `logistic_core` ve `xgb_core` ile **aynı 250 bağımsız test sorusunda** karşılaştırmak.

- **Eski Stage 08–16 dosyalarına yazma yok.** Yeni raporlar yalnızca `results/midi2026_revision/v1/01_self_verification_baseline/` ve `manifests/` altına gider.
- **LLM inference yok.** Yalnızca her model için 970 Clean-Dev satırında 2 yeni tek-sinyal kalibratör fit edilir: `platt_verification` (Stage 10 ile aynı Logistic Regression + StandardScaler) ve `isotonic_verification` (Stage 10 ile aynı Isotonic Regression).
- Eski `raw_verification`, `train_prevalence`, `logistic_core`, `xgb_core` tahminleri **Stage 14'te dondurulmuş dosyadan birebir okunur**. Yeniden fit edilmez.
- Testin 250 soru ID'si Stage 14 dondurulmuş tahminlerden türetilir; 970 Clean-Dev ile kesişim yasaktır.
- **Araştırma statüsü:** Hakem yorumlarından sonra planlanmış *post-review exploratory* analizi. Stage 12'de önceden sabitlenen birincil karşılaştırmaymış gibi sunulamaz. Test etiketleri yalnızca yeni tahminler kaydedildikten sonra değerlendirme hücresinde okunur.
- **İşleyiş:** Hücreleri yukarıdan aşağıya çalıştır. Aynı `v1` dosyaları varsa içeriği eşleşmeden üstüne yazılmaz.

In [ ]:
# 1 — Colab ortamı / Drive kurulumu (LLM çalıştırmaz)
import sys, subprocess, importlib.util
for pkg in ['pyarrow','sklearn']:
    if importlib.util.find_spec(pkg) is None:
        pip_name = 'pyarrow>=15,<25' if pkg=='pyarrow' else 'scikit-learn>=1.5,<1.9'
        subprocess.check_call([sys.executable,'-m','pip','install','-q',pip_name])

from pathlib import Path
import os, json, hashlib, math
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, average_precision_score

INPUT_MODE = 'drive'   # 'drive' (önerilir) veya 'local'
LOCAL_ROOT = Path('/content')
if INPUT_MODE=='drive':
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT=Path('/content/drive/MyDrive/LLM_Confidence_Thesis')
elif INPUT_MODE=='local':
    ROOT=LOCAL_ROOT
else:
    raise ValueError('INPUT_MODE sadece drive/local olabilir')
assert ROOT.is_dir(), ROOT

DEV_VERSION='validated_v007'; TEST_VERSION='validated_v012_final250_v1'
REV=ROOT/'results'/'midi2026_revision'/'v1'
OUT=REV/'01_self_verification_baseline'
MANIFEST_DIR=REV/'manifests'
STAGE16=REV/'00_integrity_audit'/'audit_summary_v1.json'
CORE=['mean_logprob','min_logprob','mean_token_entropy_normalized',
      'output_content_token_count','self_confidence_scaled',
      'verification_p_correct','self_consistency','unknown_response']
KEY=['model_id','question_id']; TARGET='manual_is_correct'; SIGNAL='verification_p_correct'
NEW_METHODS=['platt_verification','isotonic_verification']
ORIGINAL_METHODS=['train_prevalence','raw_verification','logistic_core','xgb_core']
METHODS=ORIGINAL_METHODS+NEW_METHODS
BOOT_REPS=4000; BOOT_SEED=20260921
EPS=1e-6; BINS=10

P={
 'dev_extended': ROOT/'features'/'assembled'/DEV_VERSION/'extended_model_matrix_v2_repair1.parquet',
 'dev_schema':ROOT/'features'/'assembled'/DEV_VERSION/'feature_schema_v1.json',
 'dev_manifest':ROOT/'features'/'assembled'/DEV_VERSION/'stage08_manifest_v2_repair1.json',
 'test_core':ROOT/'features'/'assembled'/TEST_VERSION/'core_model_matrix_v1.parquet',
 'test_manifest':ROOT/'features'/'assembled'/TEST_VERSION/'stage08_manifest_v1.json',
 'frozen_predictions':ROOT/'results'/'stage14'/TEST_VERSION/'final_test_predictions_v1.parquet',
 'predictions_manifest':ROOT/'results'/'stage14'/TEST_VERSION/'stage14_predictions_manifest_v1.json',
 'published_metrics':ROOT/'results'/'stage14'/TEST_VERSION/'final_test_metrics_v1.csv',
 'stage16_summary':STAGE16,
}
NEW_PRED=OUT/'stage17_single_signal_predictions_v1.parquet'
PRED_MAN=MANIFEST_DIR/'stage17_prediction_manifest_v1.json'
EVAL_METRICS=OUT/'stage17_final_test_metrics_v1.csv'
EVAL_BOOT=OUT/'stage17_paired_brier_bootstrap_v1.csv'
EVAL_SELECT=OUT/'stage17_selective_risk_v1.csv'
FIT_AUDIT=OUT/'stage17_development_fit_audit_v1.json'
REPORT=OUT/'stage17_run_report_v1.md'
EVAL_MAN=MANIFEST_DIR/'stage17_evaluation_manifest_v1.json'

def require(ok,msg):
    if not bool(ok): raise RuntimeError(msg)
def digest(path):
    path=Path(path)
    require(path.is_file(),f'Eksik dosya: {path}')
    h=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()
def readjson(path):return json.loads(Path(path).read_text(encoding='utf-8'))
def jsonbytes(x):return (json.dumps(x,sort_keys=True,indent=2,ensure_ascii=False,allow_nan=False)+'\n').encode('utf-8')
def immutable_bytes(path,data):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists():
        require(path.read_bytes()==data,f'IMMUTABILITY CONFLICT: {path}. v2 oluşturun.')
        print('UNCHANGED:',path.name)
    else:
        with path.open('xb') as f:f.write(data)
        print('CREATED:',path.name)
    return digest(path)
def freeze_csv(path,frame):
    return immutable_bytes(path,frame.to_csv(index=False,float_format='%.15g',lineterminator='\n').encode('utf-8'))
def freeze_json(path,obj):return immutable_bytes(path,jsonbytes(obj))
def freeze_parquet(path,frame):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists():
        old=pd.read_parquet(path)
        pd.testing.assert_frame_equal(old.reset_index(drop=True),frame.reset_index(drop=True),check_exact=True)
        print('UNCHANGED:',path.name)
    else:
        temp=path.with_name(path.name+'.tmp.parquet')
        require(not temp.exists(),f'Önce artakalan geçici dosyayı inceleyin: {temp}')
        frame.to_parquet(temp,index=False,engine='pyarrow')
        os.replace(temp,path)
        print('CREATED:',path.name)
    return digest(path)
print('ROOT:',ROOT,'\nOUT:',OUT)

In [ ]:
# 2 — Stage 16 PASS + SHA + dondurulmuş dosya sözleşmeleri
for k,p in P.items():require(p.is_file(),f'Girdi yok: {k}: {p}')
s16=readjson(STAGE16)
require(s16['status']=='PASS','Stage16 PASS değil; Stage17 çalıştırılamaz')
require(s16['required_unresolved_checks']==[],'Stage16 çözülmemiş kontrol var')
for name,sha in s16['input_sha256'].items():
    if name in P:
        require(digest(P[name])==sha,f'Stage16 sonrası kaynak değiştirilmiş: {name}')

mdev=readjson(P['dev_manifest']);mtest=readjson(P['test_manifest'])
pm=readjson(P['predictions_manifest']);sdev=readjson(P['dev_schema'])
require(mdev['dataset_version']==DEV_VERSION,'Geliştirme sürümü')
require(mtest['dataset_version']==TEST_VERSION,'Test sürümü')
require(mdev['core_features']==CORE and mtest['core_features']==CORE and sdev['core_features']==CORE,'CORE sırası uyuşmuyor')
require(pm['status']=='PREDICTIONS_FROZEN_BEFORE_TEST_LABEL_EVALUATION','Stage14 freeze status')
require(pm['predictions_sha256']==digest(P['frozen_predictions']),'Stage14 dondurulmuş tahmin SHA')
require(pm['training_n_per_model']==970 and pm['n_independent_test_questions']==250,'Stage14 adet sözleşmesi')
require(set(pm['methods'])==set(ORIGINAL_METHODS),'Stage14 yöntemleri')
require(digest(P['dev_extended'])==mdev['outputs']['extended_model_matrix_sha256'],'Stage08 dev matrix SHA')
require(digest(P['test_core'])==mtest['outputs']['core_model_matrix_sha256'],'Stage08 test core SHA')
print('STAGE 16 / STAGE 08 / STAGE 14 PROVENANCE: PASS')

In [ ]:
# 3 — Yalnız Clean-Dev etiketleri; testten sadece predictor sütunları
cols_dev=KEY+['exclude_from_final_test','used_in_model_selection',TARGET,SIGNAL]
dev_all=pd.read_parquet(P['dev_extended'],columns=cols_dev)
require(len(dev_all)==3000 and not dev_all.duplicated(KEY).any(),'Dev key/row contract')
train=dev_all.loc[dev_all['exclude_from_final_test'].astype(int).eq(0)].copy()
require(len(train)==2910 and train.groupby('model_id').size().eq(970).all(),'970/model')
require(not train['used_in_model_selection'].astype(bool).any(),'Stage03 kullanılmış sorular train içinde')
require(train[TARGET].notna().all() and train[TARGET].isin([0,1]).all(),'Dev etiketleri')
require(train[SIGNAL].notna().all() and train[SIGNAL].between(0,1).all(),'Dev verification sinyali')

frozen=pd.read_parquet(P['frozen_predictions'])
require(len(frozen)==3000 and not frozen.duplicated(KEY+['method']).any(),'Stage14 prediction rows')
require(set(frozen['method'])==set(ORIGINAL_METHODS),'Stage14 yöntemleri beklenen değil')
require(frozen.groupby(['model_id','method']).size().eq(250).all(),'250/model-method')
test_ids=set(frozen['question_id'])
require(len(test_ids)==250,'250 test ID')
require(set(train['question_id']).isdisjoint(test_ids),'DEV/TEST ID sızıntısı')
models=sorted(train['model_id'].unique())
require(set(models)==set(frozen['model_id']),'model kimliği uyuşmuyor')

# TEST LABELİ BURADA YÜKLENMİYOR.
test_full=pd.read_parquet(P['test_core'],columns=KEY+[SIGNAL])
test=test_full.loc[test_full['question_id'].isin(test_ids)].copy()
require(len(test)==750 and not test.duplicated(KEY).any(),'Test 750 model-answer')
require(set(map(tuple,test[KEY].to_numpy()))==set(map(tuple,frozen[KEY].drop_duplicates().to_numpy())),'Test pair uyuşmazlığı')
require(test[SIGNAL].notna().all() and test[SIGNAL].between(0,1).all(),'Test verification skorları')
raw=frozen.loc[frozen['method'].eq('raw_verification'),KEY+['p_correct']]
raw_join=raw.merge(test,on=KEY,validate='one_to_one')
require(np.allclose(raw_join['p_correct'],raw_join[SIGNAL],rtol=0,atol=1e-12),'Stage14 raw verification değişmiş')
print('CLEAN DEV:',len(train),'| FINAL TEST predictors:',len(test),'| MODEL:',models)
print('Final test correctness etiketleri hâlâ açılmadı.')

In [ ]:
# 4 — Eksik iki kalibratörün 970 Clean-Dev üzerinde fit edilmesi;
#     yeni tahminleri test etiketini görmeden ayrı dosyaya dondur
if NEW_PRED.exists() or PRED_MAN.exists():
    require(NEW_PRED.exists() and PRED_MAN.exists(),'Eksik/kısmi Stage17 prediction freeze')
    prev=readjson(PRED_MAN)
    require(prev['predictions_sha256']==digest(NEW_PRED),'Stage17 mevcut tahmin SHA uyuşmazlığı')
    require(prev['source_frozen_sha256']==digest(P['frozen_predictions']),'Stage14 kaynak değişmiş')
    require(prev['source_dev_sha256']==digest(P['dev_extended']),'Stage08 development değişmiş')
    newly=pd.read_parquet(NEW_PRED)
    fit_audit=readjson(FIT_AUDIT) if FIT_AUDIT.is_file() else None
    require(fit_audit is not None,'Stage17 fit audit eksik')
    print('FROZEN Stage17 tahminleri tekrar kullanılıyor; kalibratör fit edilmedi.')
else:
    new_rows=[];fit_audit={'revision_status':'post_review_exploratory',
       'per_model':[],
       'method_definitions':{
        'platt_verification':'Pipeline(StandardScaler, LogisticRegression(C=1.0,solver=lbfgs,max_iter=3000,random_state=17))',
        'isotonic_verification':'IsotonicRegression(y_min=0,y_max=1,increasing=True,out_of_bounds=clip)'},
       'train_n_per_model':970,'test_n_per_model':250,'uses_test_labels_for_fitting':False}
    for mid in models:
        d=train.loc[train['model_id']==mid].sort_values('question_id',kind='mergesort').reset_index(drop=True)
        t=test.loc[test['model_id']==mid].sort_values('question_id',kind='mergesort').reset_index(drop=True)
        y=d[TARGET].to_numpy(dtype=int)
        require(len(d)==970 and len(t)==250 and len(set(y))==2,'Model train/test adet ve etiketler')
        x=d[[SIGNAL]].to_numpy(dtype=float);xt=t[[SIGNAL]].to_numpy(dtype=float)
        platt=Pipeline([('scaler',StandardScaler()),('logistic',LogisticRegression(C=1.0,solver='lbfgs',max_iter=3000,random_state=17))])
        platt.fit(x,y)
        iso=IsotonicRegression(y_min=0.,y_max=1.,increasing=True,out_of_bounds='clip')
        iso.fit(d[SIGNAL].to_numpy(dtype=float),y)
        out={'platt_verification':platt.predict_proba(xt)[:,1],
             'isotonic_verification':iso.predict(t[SIGNAL].to_numpy(dtype=float))}
        for method,vec in out.items():
            require(len(vec)==250 and np.isfinite(vec).all() and np.all((vec>=0)&(vec<=1)),f'Tahmin aralığı: {mid} {method}')
            for q,prob in zip(t['question_id'].astype(str),vec):
                new_rows.append({'model_id':mid,'question_id':q,'method':method,'p_correct':float(prob)})
        fit_audit['per_model'].append({'model_id':mid,'n_train':970,'n_test':250,'n_positive_train':int(y.sum()),
                                       'train_prevalence':float(y.mean()),'platt_coefficient':float(platt.named_steps['logistic'].coef_[0,0]),
                                       'platt_intercept':float(platt.named_steps['logistic'].intercept_[0]),
                                       'isotonic_unique_thresholds':int(len(iso.X_thresholds_))})
        print('FIT:',mid,'(2 tek-sinyal kalibratör; LLM inference yok)')
    newly=pd.DataFrame(new_rows).sort_values(['model_id','method','question_id'],kind='mergesort').reset_index(drop=True)
    require(len(newly)==1500 and not newly.duplicated(KEY+['method']).any(),'Stage17 yeni tahmin sayısı')
    pred_sha=freeze_parquet(NEW_PRED,newly)
    fit_sha=freeze_json(FIT_AUDIT,fit_audit)
    pred_manifest={'stage':'17_self_verification_single_signal_prediction','revision_version':'v1',
      'status':'POST_REVIEW_EXPLORATORY_PREDICTIONS_FROZEN_BEFORE_THIS_NOTEBOOKS_TEST_EVALUATION',
      'methods':NEW_METHODS,'predictions_path':str(NEW_PRED),'predictions_sha256':pred_sha,
      'fit_audit_sha256':fit_sha,'source_dev_sha256':digest(P['dev_extended']),
      'source_test_core_sha256':digest(P['test_core']),
      'source_frozen_sha256':digest(P['frozen_predictions']),
      'stage16_summary_sha256':digest(STAGE16),'n_per_model':250,
      'train_n_per_model':970,'test_label_used_in_training':False,
      'claim_pre_registered_before_original_final_test':False}
    freeze_json(PRED_MAN,pred_manifest)
require(len(newly)==1500 and newly.groupby(['model_id','method']).size().eq(250).all(),'Yeni tahmin dondurma sözleşmesi')
print('STAGE17 PREDICTIONS FROZEN:',NEW_PRED)
print('SHA256:',digest(NEW_PRED))

## Değerlendirme (test etiketleri yalnız aşağıdaki hücrede okunur)

Bu hücreye gelindiğinde yeni `Stage17` tahminleri kendi manifestiyle dondurulmuştur. Sonuçlar **post-review exploratory** olarak raporlanır; orijinal Stage 14 deneyinin önceden sabitlenmiş birincil karşılaştırmasını değiştirmez.

**Metrikler:** Brier ↓, ECE (10 eşit genişlikli bin) ↓, AUROC ↑, AURC ↓, AUPRC ↑, NLL ↓. Risk sıralamasında Stage 14 ile aynı tie-break (`question_id` artan) kullanılır. Paired bootstrap: 4.000 tekrar, seed 20260921, aynı 250 soru üzerinde yeniden örnekleme.

In [ ]:
# 5 — Test etiketlerini şimdi aç; 6 yöntemi aynı 250 soru üzerinde karşılaştır
pred_manifest=readjson(PRED_MAN)
require(digest(NEW_PRED)==pred_manifest['predictions_sha256'],'Stage17 prediction freeze sonrası değişiklik')
require(digest(P['frozen_predictions'])==pred_manifest['source_frozen_sha256'],'Stage14 prediction freeze değişiklik')

# Stage14 değerlendirmesi tamamlanmış olduğundan test etiketleri burada audit/hesaplama için okunur.
label_frame=pd.read_parquet(P['test_core'],columns=KEY+[TARGET])
label_frame=label_frame.loc[label_frame['question_id'].isin(test_ids)].copy()
require(len(label_frame)==750 and not label_frame.duplicated(KEY).any(),'Test label pair count')
require(label_frame[TARGET].notna().all() and label_frame[TARGET].isin([0,1]).all(),'Test label validity')
all_pred=pd.concat([frozen[KEY+['method','p_correct']],newly[KEY+['method','p_correct']]],ignore_index=True)
require(len(all_pred)==4500 and not all_pred.duplicated(KEY+['method']).any(),'6-method join keys')
joined=all_pred.merge(label_frame,on=KEY,how='left',validate='many_to_one',indicator=True)
require(joined['_merge'].eq('both').all(),'Tahmin-etiket eşleşmesi eksik')
require(joined['p_correct'].between(0,1).all() and np.isfinite(joined['p_correct']).all(),'Probability bounds')
require(joined.groupby(['model_id','method']).size().eq(250).all(),'250/model-method')

def metric_fn(ids,y,p):
    ids=np.asarray(ids,dtype=str);y=np.asarray(y,dtype=int);p=np.asarray(p,dtype=float)
    b=np.minimum((np.clip(p,0.,1.)*BINS).astype(int),BINS-1)
    ece=sum(float(np.mean(b==j))*abs(float(y[b==j].mean())-float(p[b==j].mean()))
            for j in range(BINS) if (b==j).any())
    safe=np.clip(p,EPS,1.-EPS)
    ords=np.lexsort((ids,-p));wrong=(1-y)[ords]
    risk=np.cumsum(wrong)/np.arange(1,len(y)+1)
    return ({'n':len(y),'prevalence':float(y.mean()),
      'brier':float(np.mean((p-y)**2)),'nll':float(-np.mean(y*np.log(safe)+(1-y)*np.log1p(-safe))),
      'ece':float(ece),'auroc':float(roc_auc_score(y,p)),
      'auprc':float(average_precision_score(y,p)),'aurc':float(np.mean(risk))},risk)

metrics=[];risk_rows=[];ci_rows=[]
COMPARISONS=[('raw_verification','platt_verification'),
             ('raw_verification','isotonic_verification'),
             ('platt_verification','logistic_core'),
             ('platt_verification','xgb_core'),
             ('isotonic_verification','logistic_core'),
             ('isotonic_verification','xgb_core')]
COVERAGES=[0.2,0.5,0.8,1.0]
for mid,part in joined.groupby('model_id',sort=True):
    wide=part.pivot(index='question_id',columns='method',values='p_correct').sort_index()
    labs=part[['question_id',TARGET]].drop_duplicates('question_id').set_index('question_id').sort_index()
    require(wide.index.equals(labs.index) and len(wide)==250,'Index eşleşmesi')
    ids=wide.index.to_numpy(dtype=str);y=labs[TARGET].to_numpy(dtype=int)
    require(set(wide.columns)==set(METHODS),'Yöntem kümesi')
    for method in METHODS:
        m,curve=metric_fn(ids,y,wide[method].to_numpy(dtype=float))
        metrics.append({'model_id':mid,'method':method,**m})
        for coverage in COVERAGES:
            k=max(1,int(math.ceil(coverage*250)))
            risk_rows.append({'model_id':mid,'method':method,'coverage':coverage,'n_selected':k,
                              'risk':float(curve[k-1])})
    # YALNIZCA önceden bu hücrede yazılan karşılaştırmalar: test skorlarına göre seçilmez.
    samples=np.random.default_rng(BOOT_SEED).integers(0,250,size=(BOOT_REPS,250),endpoint=False)
    for reference,alternative in COMPARISONS:
        a=wide[alternative].to_numpy(dtype=float);r=wide[reference].to_numpy(dtype=float)
        delta=(a-y)**2-(r-y)**2
        bs=delta[samples].mean(axis=1)
        low,high=np.quantile(bs,[0.025,0.975])
        ci_rows.append({'model_id':mid,'reference':reference,'alternative':alternative,
          'comparison':f'{alternative}_minus_{reference}',
          'effect_brier':float(delta.mean()),'ci95_low':float(low),'ci95_high':float(high),
          'bootstrap_B':BOOT_REPS,'seed':BOOT_SEED,'n_questions':250,
          'interval':'post_review_exploratory_paired_question_percentile_95',
          'negative_delta_favors':'alternative'})
met=pd.DataFrame(metrics).sort_values(['model_id','method']).reset_index(drop=True)
selective=pd.DataFrame(risk_rows).sort_values(['model_id','method','coverage']).reset_index(drop=True)
boot=pd.DataFrame(ci_rows).sort_values(['model_id','reference','alternative']).reset_index(drop=True)
require(len(met)==18 and len(selective)==72 and len(boot)==18,'Output expected row count')

# Eski dört yöntem Stage14 sayılarını birebir üretmeli; birincil çalışmayı değiştirmiyoruz.
orig=pd.read_csv(P['published_metrics'])
recon=met.loc[met['method'].isin(ORIGINAL_METHODS)]
matched=recon.merge(orig,on=['model_id','method'],validate='one_to_one',suffixes=('_new','_stage14'))
require(len(matched)==12,'Original method audit 12 rows')
for c in ['n','prevalence','brier','nll','ece','auroc','auprc','aurc']:
    err=float(np.max(np.abs(matched[c+'_new']-matched[c+'_stage14'])))
    require(err<2e-9,f'Stage14 metric deviation: {c} {err}')
print('REPRODUCED ORIGINAL STAGE14: PASS')
print(met[['model_id','method','brier','ece','auroc','aurc']].to_string(index=False,float_format=lambda v:f'{v:.6f}'))
print('New prediction rows:',len(newly),'| Full comparative rows:',len(all_pred))

In [ ]:
# 6 — Ayrı revizyon klasörüne idempotent raporlar, çift SHA manifest
metrics_sha=freeze_csv(EVAL_METRICS,met)
boot_sha=freeze_csv(EVAL_BOOT,boot)
select_sha=freeze_csv(EVAL_SELECT,selective)

lines=['# MIDI2026 Stage17 — Self-verification baseline evaluation',
       '', '**Status:** PASS', '',
       '**Nature:** post-review exploratory analysis; not part of original Stage12 pre-registered primary comparison.',
       '', '- 3 models × 970 Clean-Dev training examples per model',
       '- 3 models × 250 independent test questions',
       '- New calibrators: `platt_verification`, `isotonic_verification` (single input feature only)',
       '- Reused, not refitted: Stage14 `train_prevalence`, `raw_verification`, `logistic_core`, `xgb_core`',
       '- LLM calls: 0', '',
       '## Brier score (lower is better)', '',
       '| Model | Raw | Platt | Isotonic | Logistic CORE | XGBoost CORE |',
       '|---|---:|---:|---:|---:|---:|']
for mid in models:
    rows=met.loc[met['model_id'].eq(mid)].set_index('method')
    vals=[rows.at[m,'brier'] for m in ['raw_verification','platt_verification','isotonic_verification','logistic_core','xgb_core']]
    lines.append('| '+mid+' | '+' | '.join(f'{v:.4f}' for v in vals)+' |')
lines+=['', '## Interpretation guidance',
 '- `platt_verification` vs `raw_verification`: how much does one-feature scale correction improve Brier?',
 '- `logistic_core`/`xgb_core` vs `platt_verification`: do 8 features bring added value beyond single-feature calibration?',
 '- Paired 95% CIs are descriptive/exploratory; multiple comparisons have not been adjusted.',
 '- Do not infer increased answer accuracy: only post-hoc confidence predictions change.',
 '', '## Source SHA256',
 '- Stage14 frozen predictions: `'+digest(P['frozen_predictions'])+'`',
 '- Stage17 new predictions: `'+digest(NEW_PRED)+'`',
 '', '## Files',
 '- `stage17_final_test_metrics_v1.csv`',
 '- `stage17_paired_brier_bootstrap_v1.csv`',
 '- `stage17_selective_risk_v1.csv`',
 '- `stage17_single_signal_predictions_v1.parquet`',
 '- `stage17_development_fit_audit_v1.json`', '']
report_sha=immutable_bytes(REPORT,('\n'.join(lines)+'\n').encode('utf-8'))
manifest={
 'stage':'17_midi2026_self_verification_baseline',
 'revision_version':'v1',
 'status':'PASS',
 'study_status':'post_review_exploratory',
 'n_models':len(models),'n_training_per_model':970,'n_final_questions_per_model':250,
 'new_methods':NEW_METHODS,'source_methods':ORIGINAL_METHODS,
 'new_predictions_sha256':digest(NEW_PRED),
 'stage17_prediction_manifest_sha256':digest(PRED_MAN),
 'stage16_summary_sha256':digest(STAGE16),
 'source_stage14_predictions_sha256':digest(P['frozen_predictions']),
 'outputs_sha256':{
   'metrics':metrics_sha,'paired_bootstrap':boot_sha,'selective_risk':select_sha,
   'report':report_sha,'fit_audit':digest(FIT_AUDIT)},
 'source_files_unchanged':True,
 'llm_inference_performed':False,
 'calibrators_fit_per_model':2,
 'primary_stage14_claim_unchanged':True,
 'bootstrap':{'B':BOOT_REPS,'seed':BOOT_SEED,'interval':'paired_percentile_95','multiple_comparison_adjustment':False}
}
freeze_json(EVAL_MAN,manifest)
print('\nSTAGE17 STATUS: PASS')
print('Please upload:')
for p in [EVAL_METRICS,EVAL_BOOT,EVAL_SELECT,FIT_AUDIT,REPORT,EVAL_MAN]:print(' -',p)